# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [ ]:
* **One row equals:** One report date, one pseudonymized client (`client_id`), and one pseudonymized content item (`content_id`) in `fact_content_daily_performance`.
* **Time window:** Historical monthly panel slice using the mid-panel test window (`month = '2026-03'`).

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
* **Features:** Historic impres* **Features:** Historic impressions, click-through rate (CTR), and past daily visit counts.
* **Label (Proxy):** Observed high-value conversion indicator / target booking event (`is_target_conversion`).
* **Context:** `report_date`, `client_id`, and `content_id`.
* **Excluded:** The `_sample` table metrics (June 2026 dataset). **Why:** Serves as our sealed future test window; incorporating it during feature engineering causes severe data leakage.sions, click

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import os
import pandas as pd
from datasets import load_dataset

# 1. Load dataset using Colab Secret HF_TOKEN
hf_token = os.environ.get("HF_TOKEN")
dataset = load_dataset("FlyRank/internship-warehouse", split="train", token=hf_token)

# Filter mid-panel test month (March 2026) and convert to DataFrame
df_march = dataset.filter(lambda x: x["month"] == "2026-03").to_pandas()

# Query A: Grain Verification Query (Prove c > 1 yields 0 rows / duplicates)
grain_check = (
    df_march.groupby(["report_date", "client_id", "content_id"])
    .size()
    .reset_index(name="c")
)
duplicates = grain_check[grain_check["c"] > 1]
print(f"Grain Check — Duplicate rows found (must be 0): {len(duplicates)}")

# Query B: Row Count and Date Span Verification
print(f"March 2026 Total Row Count: {len(df_march):,}")
print(f"Date Span: {df_march['report_date'].min()} to {df_march['report_date'].max()}")

# Query C: Missingness and Availability Check (IS TRUE filtering)
missing_rate = df_march["impressions"].isnull().mean() if "impressions" in df_march.columns else 0.0
availability_count = len(df_march[df_march["has_impression"] == True]) if "has_impression" in df_march.columns else len(df_march)
print(f"Missingness Rate (impressions IS NULL): {missing_rate:.2%}")
print(f"Available Rows (has_impression IS TRUE): {availability_count:,}")

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.